# MILO - Cell type composition analysis

Publication [Dann et al, 2021](https://www.nature.com/articles/s41587-021-01033-z)  
Marioni Lab [GitHub](https://github.com/MarioniLab/miloR)  

Theis lab python adaption with [pertpy](https://github.com/theislab/pertpy)

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # miloR
            library(miloR), 
            library(ggbeeswarm), 
            
            # SingleCellExperiment
            library(SingleCellExperiment), 

            # Data 
            library(tidyverse), 
            library(data.table), 

            # Plotting 
            library(patchwork), 
            library(cowplot), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv="p.3.8.12-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

In [ ]:
# Parallel 
BPPARAM <- SerialParam()
register(BPPARAM)

In [ ]:
# Reticulate python modules
sc <- import("scanpy", as="sc", convert=FALSE)
ad <- import("anndata", as="ad", convert=FALSE)

# Import Seurat object

In [ ]:
# Import Seurat object 
so <- readRDS("data/object/pp_subset.rds") # Important to load the same data as in the milo subsets to compute cell size and samples

# Time point analysis 

## Import Milo object

In [ ]:
milo_1 <- readRDS(paste0("data/object/milo/time_point/milo_k10_1.rds"))
milo_2 <- readRDS(paste0("data/object/milo/time_point/milo_k10_2.rds"))
milo_3 <- readRDS(paste0("data/object/milo/time_point/milo_k30_3.rds"))
milo_4 <- readRDS(paste0("data/object/milo/time_point/milo_k30_4.rds"))
milo_5 <- readRDS(paste0("data/object/milo/time_point/milo_k30_5.rds"))

## Import Milo results 

In [ ]:
res_1 <- readRDS("data/object/milo/time_point/res_1.rds")[[1]]
res_2 <- readRDS("data/object/milo/time_point/res_2.rds")[[1]]
res_3 <- readRDS("data/object/milo/time_point/res_3.rds")[[3]]
res_4 <- readRDS("data/object/milo/time_point/res_4.rds")[[3]]
res_5 <- readRDS("data/object/milo/time_point/res_5.rds")[[3]]

In [ ]:
contrast <- c("Tx", "D14", "D100", "GVHD")

In [ ]:
res_blood_immune <- lapply(seq_along(contrast), function(i) {rbind(res_1[[i]], res_3[[i]])})
res_skin_immune <- lapply(seq_along(contrast), function(i) {rbind(res_2[[i]], res_4[[i]])})
res_skin_structural <- res_5

In [ ]:
dabp <- function(res) {
    
    alpha <- 0.1
    group.by <- "cell_type_leiden_subset"
    
    # Beeswarm 
    data_1 <- suppressMessages(plotDAbeeswarm(res[[1]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="Tx")
    data_2 <- suppressMessages(plotDAbeeswarm(res[[2]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="D14")
    data_3 <- suppressMessages(plotDAbeeswarm(res[[3]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="D100") 
    data_4 <- suppressMessages(plotDAbeeswarm(res[[4]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="GVHD")
                               
    data <- rbind(data_1, data_2, data_3, data_4)
    data$group <- factor(data$group, levels=c("Tx", "D14", "D100", "GVHD"))
    
    data <- data %>%
        mutate(is_signif = ifelse(SpatialFDR < alpha, 1, 0)) %>%
        mutate(logFC_color = ifelse(is_signif==1, logFC, NA)) %>%
        arrange(group_by) %>%
        mutate(Nhood=factor(Nhood, levels=unique(Nhood))) %>%
        mutate(pos_x = pos_x, pos_y=pos_y) 
    
    y_breaks <- c(-floor(max(abs(data$logFC))), -floor(max(abs(data$logFC)))/2, 0, floor(max(abs(data$logFC)))/2, floor(max(abs(data$logFC))))
    n_groups <- length(unique(data$group_by))
    
    dabp <- ggplot(data, aes(pos_x, pos_y, color=logFC_color)) +
        scale_color_gradient2() +
        guides(color="none") +
        xlab("") + ylab("Log Fold Change") +
        scale_x_discrete(labels=parse(text=setNames(levels(data$group_by), seq(1,n_groups)))) + 
        scale_y_continuous(breaks=y_breaks, labels=y_breaks) +
        geom_hline(yintercept=0, size=1) + geom_point() + stat_summary(aes(x=group_by, y=logFC), fun.y=median, geom="point", size=2, color="black") +
        coord_flip() +
        facet_grid(~group) + 
        theme(axis.text.y=element_text(vjust=0.5, hjust=1)) + 
        theme_global_set(1)
                               
    return(dabp)
}


## Immune blood

In [ ]:
dabp_1 <- dabp(res_blood_immune)

In [ ]:
options(repr.plot.width=8, repr.plot.height=length(levels(dabp_1$data$group_by))*0.5)

dabp_1

## Immune skin

In [ ]:
dabp_2 <- dabp(res_skin_immune)

In [ ]:
options(repr.plot.width=8, repr.plot.height=length(levels(dabp_2$data$group_by))*0.5)

dabp_2

## Structural skin

In [ ]:
dabp_3 <- dabp(res_skin_structural)

In [ ]:
options(repr.plot.width=8, repr.plot.height=length(levels(dabp_3$data$group_by))*0.5)

dabp_3

# Genotype analysis 

## Import Milo object

In [ ]:
milo_1 <- readRDS(paste0("data/object/milo/genotype_class/milo_k10_1.rds"))
milo_2 <- readRDS(paste0("data/object/milo/genotype_class/milo_k10_2.rds"))
milo_3 <- readRDS(paste0("data/object/milo/genotype_class/milo_k30_3.rds"))
milo_4 <- readRDS(paste0("data/object/milo/genotype_class/milo_k30_4.rds"))

## Import Milo results 

In [ ]:
res_1 <- readRDS("data/object/milo/genotype_class/res_1.rds")[[1]]
res_2 <- readRDS("data/object/milo/genotype_class/res_2.rds")[[1]]
res_3 <- readRDS("data/object/milo/genotype_class/res_3.rds")[[3]]
res_4 <- readRDS("data/object/milo/genotype_class/res_4.rds")[[3]]

In [ ]:
contrast <- c("D14", "D100", "GVHD")

In [ ]:
res_blood_immune <- lapply(seq_along(contrast), function(i) {rbind(res_1[[i]], res_3[[i]])})
res_skin_immune <- lapply(seq_along(contrast), function(i) {rbind(res_2[[i]], res_4[[i]])})

In [ ]:
dabp <- function(res) {
    
    alpha <- 0.1
    group.by <- "cell_type_leiden_subset"
    
    # Beeswarm 
    data_1 <- suppressMessages(plotDAbeeswarm(res[[1]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="D14")
    data_2 <- suppressMessages(plotDAbeeswarm(res[[2]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="D100")
    data_3 <- suppressMessages(plotDAbeeswarm(res[[3]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="GVHD") 
                               
    data <- rbind(data_1, data_2, data_3)
    data$group <- factor(data$group, levels=c("D14", "D100", "GVHD"))
    
    data <- data %>%
        mutate(is_signif = ifelse(SpatialFDR < alpha, 1, 0)) %>%
        mutate(logFC_color = ifelse(is_signif==1, logFC, NA)) %>%
        arrange(group_by) %>%
        mutate(Nhood=factor(Nhood, levels=unique(Nhood))) %>%
        mutate(pos_x = pos_x, pos_y=pos_y) 
    
    y_breaks <- c(-floor(max(abs(data$logFC))), -floor(max(abs(data$logFC)))/2, 0, floor(max(abs(data$logFC)))/2, floor(max(abs(data$logFC))))
    n_groups <- length(unique(data$group_by))
    
    dabp <- ggplot(data, aes(pos_x, pos_y, color=logFC_color)) +
        scale_color_gradient2() +
        guides(color="none") +
        xlab("") + ylab("Log Fold Change") +
        scale_x_discrete(labels=parse(text=setNames(levels(data$group_by), seq(1,n_groups)))) + 
        scale_y_continuous(breaks=y_breaks, labels=y_breaks) +
        geom_hline(yintercept=0, size=1) + geom_point() + stat_summary(aes(x=group_by, y=logFC), fun.y=median, geom="point", size=2, color="black") +
        coord_flip() +
        facet_grid(~group) + 
        theme(axis.text.y=element_text(vjust=0.5, hjust=1)) + 
        theme_global_set(1)
                               
    return(dabp)
    
}

## Immune blood

In [ ]:
dabp_1 <- dabp(res_blood_immune)

In [ ]:
options(repr.plot.width=8, repr.plot.height=length(levels(dabp_1$data$group_by))*0.5)

dabp_1

## Immune skin

In [ ]:
dabp_2 <- dabp(res_skin_immune)

In [ ]:
options(repr.plot.width=8, repr.plot.height=length(levels(dabp_2$data$group_by))*0.5)

dabp_2

# Session info

In [ ]:
sessionInfo()